# Progetto: Web Scraper per Libreria Online 
## Obiettivo

Estrarre dati strutturati (Titolo, Rating, Prezzo, Disponibilità) dal sito `books.toscrape.com`, gestendo la paginazione e salvando i risultati in un file CSV. Questo progetto è stato strutturato per dimostrare pratiche di sviluppo software robuste e manutenibili, adatte a un contesto professionale.

### Tecnologie Utilizzate
- **Python 3**
- **Requests**: Per le richieste HTTP.
- **BeautifulSoup4**: Per il parsing dell'HTML.
- **Pandas**: Per la manipolazione e l'esportazione dei dati.
- **Logging**: Per un tracciamento degli eventi più strutturato rispetto a `print()`.

## 1. Setup dell'Ambiente e Import delle Librerie

Le importazioni sono raggruppate all'inizio per chiarezza. Introduciamo il modulo `logging` per una gestione degli errori e dello stato più professionale e configurabile, e `urljoin` per costruire URL in modo sicuro.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import logging
from urllib.parse import urljoin
import re

# --- Configurazione del Logging ---
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler() # Stampa i log sulla console
    ]
)

## 2. Definizione delle Funzioni Core

Il codice è stato scomposto in funzioni più piccole e specifiche (Principio di Singola Responsabilità). Questo rende il codice:
- **Più leggibile:** Ogni funzione ha un solo scopo.
- **Più testabile:** È possibile testare ogni unità di logica in isolamento.
- **Più manutenibile:** Una modifica al parsing del prezzo, per esempio, richiederà di toccare solo una piccola funzione.

Inoltre, vengono usati **type hints** (es. `url: str`) e **docstrings** per documentare il comportamento di ogni funzione, pratiche standard in progetti professionali.

In [ ]:
def fetch_html(url: str) -> BeautifulSoup | None:
    """Effettua una richiesta GET all'URL specificato e restituisce un oggetto BeautifulSoup."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Lancia un'eccezione per status code 4xx/5xx
        return BeautifulSoup(response.content, "html.parser")
    except requests.exceptions.RequestException as e:
        logging.error(f"Errore durante la richiesta HTTP per {url}: {e}")
        return None

def parse_book_details(book_soup: BeautifulSoup) -> dict:
    """Estrae i dettagli di un singolo libro da un oggetto BeautifulSoup."""
    # Mappatura per convertire il rating testuale in numero
    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    
    # Titolo
    title_tag = book_soup.find("h3").find("a")
    title = title_tag["title"] if title_tag else "N/A"

    # Rating
    rating_tag = book_soup.find("p", class_="star-rating")
    rating_class = rating_tag["class"][1] if rating_tag and len(rating_tag.get("class", [])) > 1 else None
    rating = rating_map.get(rating_class, 0)

    # Prezzo (con pulizia del dato)
    price_tag = book_soup.find("p", class_="price_color")
    price_str = price_tag.text if price_tag else "£0.00"
    price_float = float(re.search(r'[\d\.]+', price_str).group(0)) # Estrae solo il numero

    # Disponibilità
    availability_tag = book_soup.find("p", class_="instock availability")
    availability = availability_tag.text.strip() if availability_tag else "N/A"
    
    return {
        "Title": title,
        "Rating": rating,
        "Price_Pounds": price_float,
        "Availability": availability
    }

def get_next_page_url(soup: BeautifulSoup, base_url: str) -> str | None:
    """Trova l'URL della pagina successiva, se esiste."""
    next_page_tag = soup.select_one("li.next > a")
    if next_page_tag:
        relative_url = next_page_tag["href"]
        return urljoin(base_url, relative_url)
    return None

## 3. Orchestrazione dello Scraper

 Una funzione `main()` orchestra l'intero processo. Questo è un pattern comune che rende chiaro il punto di ingresso e il flusso logico dello script. Il ciclo `while` è più pulito e la logica di paginazione è gestita in modo robusto. I dati vengono accumulati in una lista di dizionari, che è più efficiente rispetto a concatenare DataFrame in un ciclo.

In [ ]:
def scrape_all_books(start_url: str) -> pd.DataFrame:
    """Orchestra il processo di scraping su tutte le pagine del sito."""
    all_books_data = []
    current_url = start_url
    page_count = 1
    
    while current_url:
        logging.info(f"Scraping pagina {page_count}: {current_url}")
        soup = fetch_html(current_url)
        
        if not soup:
            logging.warning(f"Impossibile recuperare la pagina {current_url}. Interruzione.")
            break
            
        books_on_page = soup.find_all("article", class_="product_pod")
        for book_soup in books_on_page:
            book_data = parse_book_details(book_soup)
            all_books_data.append(book_data)
            
        current_url = get_next_page_url(soup, current_url)
        page_count += 1
        
    logging.info("Scraping completato.")
    return pd.DataFrame(all_books_data)

## 4. Esecuzione e Salvataggio dei Dati

**Miglioramento Senior:** L'esecuzione dello script è protetta da un blocco `if __name__ == '__main__':`. Questo è fondamentale in Python e permette di importare le funzioni di questo script in altri moduli senza eseguire automaticamente lo scraping. 

In [ ]:
if __name__ == '__main__':
    BASE_URL = "https://books.toscrape.com/catalogue/page-1.html"
    OUTPUT_FILE = "books_data.csv"

    books_df = scrape_all_books(BASE_URL)

    if not books_df.empty:
        # Visualizziamo un'anteprima dei dati
        print(books_df.head())
        print(f"\nTotale libri estratti: {len(books_df)}")
        
        # Salvataggio in CSV
        try:
            books_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
            logging.info(f"Dati salvati con successo in '{OUTPUT_FILE}'")
        except IOError as e:
            logging.error(f"Errore durante il salvataggio del file CSV: {e}")
    else:
        logging.warning("Nessun dato è stato estratto, il file CSV non sarà creato.")